# NB12 — EBM Sentiment Extension (M2.5-sent)

**Purpose:** Fill the missing cell in the 2x2 {algorithm: RF/EBM} x {features: macro/macro+sentiment}
design. Fits EBM on the full 61-feature M3 input set (41 macro + 20 sentiment), matching the exact
feature set M3 actually used — enabling a clean algorithm-only comparison against M3 (XGBoost+sentiment)
and a clean feature-only comparison against M2.5-matched (EBM, macro-only).

**This notebook is a standalone extension.** It reads pre-built datasets from NB05A/NB05B and does not
modify or retrain M2-matched, M2.5-matched, or M3. It is purely additive.

**Inputs:** `df_sent_augmented.csv`, `M3_features.csv` (from NB05A); `oos_predictions_stage2.csv`,
`oos_predictions_stage2_ebm.csv` (from NB05A/NB05B, for comparison table only — not retrained).


## Cell 1 — Imports and paths

In [ ]:
import os, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss
)

from interpret.glassbox import ExplainableBoostingClassifier

BASE     = Path(r'.')
OUT_DIR  = BASE / 'data' / 'processed' / 'augmented_analysis'
FIG_DIR  = BASE / 'figures'   # adjust if your figures live elsewhere

TARGET       = 'target_h2'
RANDOM_STATE = 42   # confirmed to match NB05A Cell 2 and NB05B Cell 2
N_BOOTSTRAP  = 1000 # confirmed to match NB05B Cell 2

print('Paths configured.')
print(f'OUT_DIR : {OUT_DIR}')
print(f'Exists  : {OUT_DIR.exists()}')


## Cell 2 — Load data and feature lists

In [ ]:
# ── Load Stage 2 dataset (2003–2020) ───────────────────────────────────────
df_sent = pd.read_csv(OUT_DIR / 'df_sent_augmented.csv')

# ── Load the ACTUAL M3 feature list (61 features: 41 macro + 20 sentiment) ─
M3_FEATURES = pd.read_csv(OUT_DIR / 'M3_features.csv', header=None)[0].tolist()

n_crisis_sent  = int(df_sent[TARGET].sum())
base_rate_sent = df_sent[TARGET].mean()

macro_n = sum(1 for f in M3_FEATURES if f.endswith('_dm'))
sent_n  = sum(1 for f in M3_FEATURES if f.endswith('_rdm'))

print('=== M2.5-sent DATASET (macro + sentiment, 2003–2020) ===')
print(f'Rows          : {len(df_sent)}')
print(f'Crisis events : {n_crisis_sent}  ({100*base_rate_sent:.2f}% base rate)')
print(f'Features      : {len(M3_FEATURES)} total  ({macro_n} macro + {sent_n} sentiment)')
print()
assert len(M3_FEATURES) == 61, f'Expected 61 features, found {len(M3_FEATURES)} — check M3_features.csv'
assert macro_n == 41 and sent_n == 20, 'Feature split does not match expected 41/20 — investigate before proceeding'
print('✅  Feature count verified: 61 (41 macro + 20 sentiment) — matches actual M3 training set.')

# ── Load existing OOS predictions for the comparison table (read-only) ─────
oos_s2     = pd.read_csv(OUT_DIR / 'oos_predictions_stage2.csv')          # has prob_m2m, prob_m3
oos_s2_ebm = pd.read_csv(OUT_DIR / 'oos_predictions_stage2_ebm.csv')      # has prob_ebm (M2.5-matched)


## Cell 3 — Expanding-window CV function (identical to NB05A/NB05B)

In [ ]:
def expanding_window_cv(df, feature_cols, target_col, model_factory,
                        min_train_years=4, hold_out_years=(2019, 2020)):
    """Expanding-window cross-validation.

    Identical implementation to NB05A/NB05B — ensures M2.5-sent metrics
    are directly comparable to M2-matched, M2.5-matched, and M3.
    """
    cv_years   = sorted(y for y in df['year'].unique() if y not in hold_out_years)
    first_eval = cv_years[min_train_years]
    eval_years = [y for y in cv_years if y >= first_eval]
    print(f'  Burn-in: {cv_years[0]:.0f}–{first_eval-1:.0f} | '
          f'Eval: {eval_years[0]:.0f}–{eval_years[-1]:.0f} ({len(eval_years)} folds)')

    oos_idx, oos_prob, fold_records = [], [], []
    fold_auprcs = []

    for yr in eval_years:
        tr = df['year'] < yr
        te = df['year'] == yr
        X_tr, y_tr = df.loc[tr, feature_cols], df.loc[tr, target_col]
        X_te, y_te = df.loc[te, feature_cols], df.loc[te, target_col]
        if y_tr.sum() == 0 or len(X_te) == 0:
            continue

        n_neg_fold = (y_tr == 0).sum()
        n_pos_fold = max((y_tr == 1).sum(), 1)
        fold_spw   = n_neg_fold / n_pos_fold

        mdl = model_factory(fold_spw)
        mdl.fit(X_tr, y_tr)

        prob = mdl.predict_proba(X_te)[:, 1]
        oos_idx.extend(df.index[te].tolist())
        oos_prob.extend(prob.tolist())

        if y_te.sum() > 0:
            fold_auprc = average_precision_score(y_te, prob)
            fold_auprcs.append(fold_auprc)
        else:
            fold_auprc = float('nan')

        fold_records.append({
            'year': yr, 'n_train': int(tr.sum()), 'crises_train': int(y_tr.sum()),
            'n_test': int(te.sum()), 'crises_test': int(y_te.sum()),
            'fold_spw': round(fold_spw, 2), 'fold_auprc': round(fold_auprc, 4) if not np.isnan(fold_auprc) else 'n/a'
        })

    folds_df = pd.DataFrame(fold_records)
    if fold_auprcs:
        print(f'  Per-fold AUPRC (folds with crisis cases): '
              f'mean={np.mean(fold_auprcs):.4f}  std={np.std(fold_auprcs):.4f}')
    return np.array(oos_idx), np.array(oos_prob), folds_df, fold_auprcs


## Cell 4 — Bootstrap CI helper

In [ ]:
def bootstrap_metrics(y_true, y_prob, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    """Bootstrap 95% CIs for AUROC and AUPRC.

    NOTE: as with NB05A/NB05B, these CIs treat observations as i.i.d.,
    which panel dependence violates — indicative only, not formal intervals.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    rng = np.random.RandomState(seed)
    aurocs, auprcs = [], []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        aurocs.append(roc_auc_score(yt, yp))
        auprcs.append(average_precision_score(yt, yp))
    return {
        'auroc_ci': (np.percentile(aurocs, 2.5), np.percentile(aurocs, 97.5)),
        'auprc_ci': (np.percentile(auprcs, 2.5), np.percentile(auprcs, 97.5)),
    }


## Cell 5 — EBM factory with sample-weighting (identical spec to NB05B's M2.5-matched)

In [ ]:
class EBMWeightedWrapper:
    """Wraps EBM to apply inverse-class-frequency sample weights at fit time.
    Matches NB05B's ebm_weighted_factory approach exactly."""
    def __init__(self, **kwargs):
        self.model = ExplainableBoostingClassifier(**kwargs)

    def fit(self, X, y):
        y_arr = np.asarray(y)
        spw = (y_arr == 0).sum() / max((y_arr == 1).sum(), 1)
        sw = np.where(y_arr == 1, spw, 1.0)
        self.model.fit(X, y, sample_weight=sw)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)


def ebm_sent_factory(fold_spw=None):
    """IDENTICAL hyperparameters to NB05B's M2.5-matched spec —
    only the feature set changes (61 vs 41)."""
    return EBMWeightedWrapper(
        max_rounds=5000, interactions=5, learning_rate=0.01,
        min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    )


## Cell 6 — Train M2.5-sent (EBM on full 61-feature set, Stage 2)

In [ ]:
print('=== TRAINING M2.5-sent: EBM on df_sent, 61 features (41 macro + 20 sentiment) ===')
print(f'Dataset : {len(df_sent)} rows | {n_crisis_sent} target_h2 positives')
print('This may take several minutes — EBM on 61 features is slower than on 41.')
print()

idx_ebm_sent, prob_ebm_sent, folds_ebm_sent, fold_auprcs_sent = expanding_window_cv(
    df_sent, M3_FEATURES, TARGET, ebm_sent_factory
)

y_ebm_sent = df_sent.loc[idx_ebm_sent, TARGET].values

auroc_ebm_sent = roc_auc_score(y_ebm_sent, prob_ebm_sent)
auprc_ebm_sent = average_precision_score(y_ebm_sent, prob_ebm_sent)
brier_ebm_sent = brier_score_loss(y_ebm_sent, prob_ebm_sent)
ci_ebm_sent    = bootstrap_metrics(y_ebm_sent, prob_ebm_sent)

print()
print('=== M2.5-sent Results ===')
print(f'  AUROC : {auroc_ebm_sent:.4f}  95% CI [{ci_ebm_sent["auroc_ci"][0]:.3f}, {ci_ebm_sent["auroc_ci"][1]:.3f}]')
print(f'  AUPRC : {auprc_ebm_sent:.4f}  95% CI [{ci_ebm_sent["auprc_ci"][0]:.3f}, {ci_ebm_sent["auprc_ci"][1]:.3f}]')
print(f'  Brier : {brier_ebm_sent:.4f}')


## Cell 7 — Full 2×2 comparison table

In [ ]:
print('=' * 100)
print(' STAGE 2 — COMPLETE 2×2 COMPARISON: {RF, EBM} x {macro-only, macro+sentiment}')
print('=' * 100)

# Pull existing metrics from already-saved OOS predictions (NOT retrained)
def metrics_from_oos(df, prob_col, target_col=TARGET):
    sub = df.dropna(subset=[prob_col])
    y, p = sub[target_col].values, sub[prob_col].values
    return roc_auc_score(y, p), average_precision_score(y, p), brier_score_loss(y, p)

auroc_m2m, auprc_m2m, brier_m2m = metrics_from_oos(oos_s2, 'prob_m2m')
auroc_m3,  auprc_m3,  brier_m3  = metrics_from_oos(oos_s2, 'prob_m3')
auroc_ebm, auprc_ebm, brier_ebm = metrics_from_oos(oos_s2_ebm, 'prob_ebm')

rows = [
    ('RF',  'macro-only (41)',     auroc_m2m,      auprc_m2m,      brier_m2m),
    ('RF',  'macro+sentiment (61)','—  (M3 uses XGBoost, not RF)', '—', '—'),
    ('EBM', 'macro-only (41)',     auroc_ebm,      auprc_ebm,      brier_ebm),
    ('EBM', 'macro+sentiment (61)',auroc_ebm_sent, auprc_ebm_sent, brier_ebm_sent),
    ('XGB', 'macro+sentiment (61)',auroc_m3,       auprc_m3,       brier_m3),
]

print(f"{'Algorithm':<6} {'Features':<22} {'AUROC':>10} {'AUPRC':>10} {'Brier':>10}")
print('-' * 100)
for algo, feat, a, p, b in rows:
    a_s = f'{a:.4f}' if isinstance(a, float) else a
    p_s = f'{p:.4f}' if isinstance(p, float) else p
    b_s = f'{b:.4f}' if isinstance(b, float) else b
    print(f'{algo:<6} {feat:<22} {a_s:>10} {p_s:>10} {b_s:>10}')

print()
print('KEY ISOLATED COMPARISONS:')
print(f'  Algorithm effect  (61 features, EBM vs XGB): '
      f'AUPRC {auprc_ebm_sent:.4f} vs {auprc_m3:.4f}  (Δ={auprc_ebm_sent-auprc_m3:+.4f})')
print(f'  Feature effect    (EBM, macro-only vs +sentiment): '
      f'AUPRC {auprc_ebm:.4f} vs {auprc_ebm_sent:.4f}  (Δ={auprc_ebm_sent-auprc_ebm:+.4f})')


## Cell 8 — Save OOS predictions (distinct filenames, doesn't touch existing files)

In [ ]:
oos_ebm_sent_series = pd.Series(prob_ebm_sent, index=idx_ebm_sent, name='prob_ebm_sent')
oos_sent_out = df_sent[['year', 'iso', TARGET]].join(oos_ebm_sent_series)

out_path = OUT_DIR / 'oos_predictions_stage2_ebm_sent.csv'
oos_sent_out.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print(f'Rows with M2.5-sent prob: {oos_sent_out["prob_ebm_sent"].notna().sum()}')


## Cell 9 — Completion summary

In [ ]:
print('=' * 65)
print(' NB12 — EBM SENTIMENT EXTENSION (M2.5-sent) COMPLETE')
print('=' * 65)
print()
print(f'M2.5-sent AUROC : {auroc_ebm_sent:.4f}  [{ci_ebm_sent["auroc_ci"][0]:.3f}, {ci_ebm_sent["auroc_ci"][1]:.3f}]')
print(f'M2.5-sent AUPRC : {auprc_ebm_sent:.4f}  [{ci_ebm_sent["auprc_ci"][0]:.3f}, {ci_ebm_sent["auprc_ci"][1]:.3f}]')
print(f'M2.5-sent Brier : {brier_ebm_sent:.4f}')
print()
print('No existing files (NB05A/NB05B outputs) were modified.')
print('Next: paste the printed output back to Claude for review before writing into the dissertation.')
